# CONFIG

In [2]:
import os

print(os.getcwd())
if not os.getcwd().endswith("app"):
    os.chdir("../app")
    print(os.getcwd())

import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

%load_ext autoreload
%autoreload 2
%matplotlib inline

/home/turbotowerlnx/Documents/Master/LC-Movie-Genre-Prediction/notebooks
/home/turbotowerlnx/Documents/Master/LC-Movie-Genre-Prediction/app


# CODE

In [3]:

import os
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
import google.generativeai as genai
from dotenv import load_dotenv
from maikol_utils.print_utils import print_separator, print_log
from maikol_utils.file_utils import save_json

from src.config import Configuration
from src.models import (
    MovieBatchClassification,
)
from src.models import classify_movies_batch
from src.scripts import compute_metrics


[nltk_data] Downloading package stopwords to
[nltk_data]     /home/turbotowerlnx/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/turbotowerlnx/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [4]:
CONFIG = Configuration()

In [ ]:
load_dotenv()
gemini_api_key = os.getenv('GEMINI_API_KEY')

if not gemini_api_key:
    raise ValueError("GEMINI_API_KEY not found in .env file")

print_separator("CONFIGURE MODEL", sep_type="LONG")
genai.configure(api_key=gemini_api_key)

model = genai.GenerativeModel(
    'gemini-2.5-pro',
    generation_config=genai.GenerationConfig(
        response_mime_type="application/json",
        response_schema=MovieBatchClassification
    )
)
# ========================================================
#                       Load Data
# ========================================================
print_separator("LOADING DATA", sep_type="LONG")
train_df = pd.read_csv(CONFIG.train_data)
test_df = pd.read_csv(CONFIG.test_data)

print_log(f" - Train shape: {train_df.shape}")
print_log(f" - Test shape: {test_df.shape}")


# ========================================================
#                       PREDICT
# ========================================================
print_separator("PREDICTING", sep_type="LONG")
predictions = classify_movies_batch(
    model,
    df_train=train_df,
    df_test=test_df,
    batch_size=CONFIG.batch_size
)